# Cerebellar alignment tutorial

Tutorial for cerebellar alignment pipeline.

### Make sure all paths are correct before running!

## Cerebellar alignment

**Add credits**

Some individuals with cortical strokes experience cerebral tissue swelling in week 0 (W0) which goes down in W4. As a result, 

### To-do

**Cerebellum-only image coregistration**

- [x] Using SUITPy isolation pipeline, get a cerebellar isolation mask for each subject at each time point
- [ ] Get a cerebellum-only image: multiply isolation mask (binary) by T1 anatomical - save with suffix `_T1_cerebellum_only.nii` in new folder "cerebellar_alignment" for each subj-week
      *Note*: SPM `coreg` does not read archived (.nii.gz) files; cerebellar image MUST be saved as uncompressed Nifti
- [ ] Update `sc_anat.m` function to read images with suffix `_T1_cerebellum_only.nii` (save as new function, `sc_anat_cerebel.m`); update directory (add "cerebellar_alignment" to path). Run coregistration on cerebellum-only images.

**Updating affine**
- [ ] Coming soon: function to update affines
- [ ] Use that function to update the **affine** of the T1 anatomical with the affine from cerebellum-only alignment (i.e. set its affine to that of the cerebellum-only coregistered image for the corresponding week); save in "cerebellum_alignment" with suffix
- [ ] Repeat for tissue probability maps (e.g. white matter probability map, c2-prefixed file), and save in "cerebellar_alignment" folder as well.
- [ ] Run linear regression on new cerebellum-aligned-affine, save slope and intercept images to each subject's folder under "cerebellar_alignment_regression" subfolder


In [1]:
# if not in same directory as fcn (e.g. avg_vol.py), import cannot find it since notebook is not in root directory of project.
# so add project root
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

In [2]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl

from image_analysis import avg_vol as av
import cerebellum_only_image as coi
import affine_assignment

from pathlib import Path
import os

In [3]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [4]:
#coi.cerebellum_only_img?

### 1. create subfolder (for each subject) for cerebellar-only alignment

subj --> cerebellar_alignment --> [Week_num]

In [5]:
"""
# create cerebellar_alignment subfolder for each subject

for subj in p_df['subj_id'].unique():

    # make directory to store output for each subj
    results_path_dir = Path(anat_dir)/subj/'cerebellar_alignment/'

    # comment out this line after running once, for safety
    #results_path.mkdir(parents=True) # exist_ok = True

    print(f'{subj} cerebellar_alignment directory created!')
"""

"\n# create cerebellar_alignment subfolder for each subject\n\nfor subj in p_df['subj_id'].unique():\n\n    # make directory to store output for each subj\n    results_path_dir = Path(anat_dir)/subj/'cerebellar_alignment/'\n\n    # comment out this line after running once, for safety\n    #results_path.mkdir(parents=True) # exist_ok = True\n\n    print(f'{subj} cerebellar_alignment directory created!')\n"

### 2. cerebellum-only image

In [6]:
# cerebellum-only image
"""
#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # images: t1_anat, cerebel_mask (dseg)

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(mask_path).is_file():
        print(f'mask path does not exist for {subj_id} in week {week}')
        continue
    

    # make a new folder inside subject's week folder for results
    #results_path = Path(anat_dir)/subj_id/week/'cerebellar_alignment/'

    results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week

    # comment out this line after done
    results_path.mkdir(parents=True, exist_ok = True) # only make this directory once!


    # results_path: subj_id, cerebellar_alignment, Wn (create directory ONCE)

    # remove this line after running this function once!!
    #results_path.mkdir(parents=True, exist_ok = True) # only make this directory once!

    #__________________________________

    # function goes here
    coi.cerebellum_only_img(
        cerebellar_mask = mask_path,
        anat_img = t1_path,
        results_path = results_path,
        subj_id = subj_id,
        week = week
    )


    print(f'{subj_id} {week} cerebellar image done')
"""


"\n#_______________________________\n# base loop\nfor i in range(0, p_df.shape[0]):\n    p_id = p_df['ID'].iloc[i]\n    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces\n    p_centre = (str(p_df['Centre'].iloc[i])).strip()\n    refT1 = (p_df['RefT1'].iloc[i]).strip()\n\n    subj_id = f'{p_centre.strip()}_{p_id}'\n\n    # images: t1_anat, cerebel_mask (dseg)\n\n    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'\n    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'\n\n    # check that paths exist\n    if not Path(t1_path).is_file():\n        print(f'T1 path does not exist for {subj_id} in week {week}')\n        continue\n\n    if not Path(mask_path).is_file():\n        print(f'mask path does not exist for {subj_id} in week {week}')\n        continue\n    \n\n    # make a new folder inside subject's week folder for results\n    #results_path = Path(anat_dir)/subj_id/week/'cerebellar_alignment/'\n\n   

### 3. SPM coregistration of cerebelli

Use SPM coregistration (function provided in this directory) to coregister each subject week's cerebellum-only image to the reference image.

This will update each (non-reference) cerebellum-only image affine for better alignment of cerebellar voxels.

### 4. Apply new (cerebellar-only-alignment) affine to relevant images

Use the function supplied (`affine_assignment`). This will take the affine from the target image (each week's re-coregistered cerebellum-only image) and assign it to the source image (defined as required).

Save this as a new image in the `cerebellar_alignment` subfolder for each reference week.


In this case (June 15 at 1:09pm)

- T1_anatomical (native space) (for new normalization files)

- white matter probability map (native space)

In [ ]:
affine_assignment.affine_assignment?

Signature:
affine_assignment.affine_assignment(
    reference_img,
    source_img,
    results_path,
    update_sform=False,
    update_qform=False,
)
Docstring:
Inputs:
    reference image (Nifti or str): image containing target affine
    source image (Nifti or str): image with affine to update

    results_path (str): path to store image with updated affine

    # taken out for now; testing
    subj_id, week (str)

    update_sform, update_qform: False by default; updates s-form, q-form matrices of source image with that of reference image.

Assigns affine from reference/target image to source image (for world-coordinates alignment).

Output:
    updated affine source image
File:      ~/Documents/GitHub/smarts_cerebellum/cerebellar_alignment/affine_assignment.py
Type:      function

In [ ]:
stop

In [ ]:
# test this on one subject
#p_df = p_df[p_df.subj_id == 'CU_2538']

#### first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
#### and test with the regression function.

# double check the path on this before running (and maybe add suffix to this? if so, need to update suffix on all subsequent functions)

In [ ]:
def subj_week(p_df, 
              function, # function to apply

              # specify required images; at least 1
              ref_img,
              source_img = None, # e.g. for cerebellar_alignment assign_affine, need two images (target affine image, image with affine to-be-changed)

              tissue = None,
              ref_ignore = True # reference image unmodified
              ):

    # loop through each subject-week
    for i in range(0, p_df.shape[0]):
        p_id = p_df['ID'].iloc[i]
        week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
        p_centre = (str(p_df['Centre'].iloc[i])).strip()
        refT1 = (p_df['RefT1'].iloc[i]).strip()

        subj_id = f'{p_centre.strip()}_{p_id}'


        #______________________________________________________________
        # ignore ref T1 if required
        if ref_ignore == True:
            if week==refT1:
                # will still need to save the image again to the cerebellar_alignment directory
                # but if we don't skip it, the image won't change...but skip it anyways, good practice NOT to change the reference

                # save unchanged reference c2 image to cerebellar_alignment again
                c2_ref_week = f'{anat_dir}/{subj_id}/{week}/c2{subj_id}_{week}_T1.nii'
                c2_ref_week_img = nib.load(c2_ref_week)
                nib.save(c2_ref_week_img, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/c2{subj_id}_{week}_T1.nii')

                print(f"skipping reference c2 for {subj_id} but saved it again to cerebellar_alignment under reference week {week}")
                continue
            #______________________________________________________________

        tissue_dict = {
            'gm': 'c1',
            'wm': 'c2',
            'csf': 'c3'
        }
        
        if not tissue == None:
            source_img = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
        else:
            source_img = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
        
        ref_img = f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/{subj_id}_{week}_T1_cerebellum_only.nii'


        #______________________________________________________________
        # check that paths exist
        if not Path(c2_source_path).is_file():
            print(f'T1 (source) path does not exist for {subj_id} in week {week}')
            continue

        if not Path(cerebel_ref_path).is_file():
            print(f'Cerebellar (ref) path does not exist for {subj_id} in week {week}')
            continue

        results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week
        #______________________________________________________________
        
        #__________________________________
        
        # function goes here

        # first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
        # and test with the regression function.
        wm_affine_updated = affine_assignment.affine_assignment(
            reference_img = cerebel_ref_path,
            source_img = c2_source_path,

            results_path = results_path, # deprecated in function - need to update there.

            # should I also update qform? If yes, should set qform (of source image) equal to sform (of reference image) - since our reference image is the cerebellum-only image (coregistered), so it doesn't have qform
            update_sform = True # just in case
        )
        

        print(f'{subj_id} {week} wm_native affine updated!')
        nib.save(wm_affine_updated, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/c2{subj_id}_{week}_T1.nii')
    

In [ ]:

# white matter probability images: affine update

#_______________________________
# base loop

for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # images: re-coregistered cerebellum image, white matter probability image.

    #______________________________________________________________
    # ignore ref T1
    if week==refT1:
        # will still need to save the image again to the cerebellar_alignment directory
        # but if we don't skip it, the image won't change...but skip it anyways, good practice NOT to change the reference

        # save unchanged reference c2 image to cerebellar_alignment again
        c2_ref_week = f'{anat_dir}/{subj_id}/{week}/c2{subj_id}_{week}_T1.nii'
        c2_ref_week_img = nib.load(c2_ref_week)
        nib.save(c2_ref_week_img, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/c2{subj_id}_{week}_T1.nii')

        print(f"skipping reference c2 for {subj_id} but saved it again to cerebellar_alignment under reference week {week}")
        continue
    #______________________________________________________________

    
    c2_source_path = f'{anat_dir}/{subj_id}/{week}/c2{subj_id}_{week}_T1.nii'
    cerebel_ref_path = f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/{subj_id}_{week}_T1_cerebellum_only.nii'


    #______________________________________________________________
    # check that paths exist
    if not Path(c2_source_path).is_file():
        print(f'T1 (source) path does not exist for {subj_id} in week {week}')
        continue

    if not Path(cerebel_ref_path).is_file():
        print(f'Cerebellar (ref) path does not exist for {subj_id} in week {week}')
        continue

    results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week
    #______________________________________________________________
    
    #__________________________________
    
    # function goes here

    # first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
    # and test with the regression function.
    wm_affine_updated = affine_assignment.affine_assignment(
        reference_img = cerebel_ref_path,
        source_img = c2_source_path,

        results_path = results_path, # deprecated in function - need to update there.

        # should I also update qform? If yes, should set qform (of source image) equal to sform (of reference image) - since our reference image is the cerebellum-only image (coregistered), so it doesn't have qform
        update_sform = True # just in case
    )
    

    print(f'{subj_id} {week} wm_native affine updated!')
    nib.save(wm_affine_updated, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/c2{subj_id}_{week}_T1.nii')
"""

skipping reference c2 for CU_2538 but saved it again to cerebellar_alignment under reference week W0
CU_2538 W4 wm_native affine updated!


In [ ]:
# T1 anatomicals: affine update

#_______________________________
# base loop

for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # images: re-coregistered cerebellum image, t1 anatomcial image

    #______________________________________________________________
    # ignore ref T1 but save it again (as a copy) to the new directory, `cerebellar_alignment` (for consistency)
    if week==refT1:
        # will still need to save the image again to the cerebellar_alignment directory
        # but if we don't skip it, the image won't change...but skip it anyways, good practice NOT to change the reference

        # save unchanged reference t1 image to cerebellar_alignment again
        t1_ref_week = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
        t1_ref_week_img = nib.load(t1_ref_week)
        nib.save(t1_ref_week_img, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/{subj_id}_{week}_T1.nii')

        print(f"skipping reference t1 anat for {subj_id} but saved it again to cerebellar_alignment under reference week {week}")
        continue
    #______________________________________________________________

    # load source and target (affine) images (for non-ref weeks)
    t1_source_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    cerebel_ref_path = f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/{subj_id}_{week}_T1_cerebellum_only.nii'


    #______________________________________________________________
    # check that paths exist
    if not Path(t1_source_path).is_file():
        print(f'T1 (source) path does not exist for {subj_id} in week {week}')
        continue

    if not Path(cerebel_ref_path).is_file():
        print(f'Cerebellar (ref) path does not exist for {subj_id} in week {week}')
        continue

    results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week
    #______________________________________________________________
    
    #__________________________________
    
    # function goes here

    # first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
    # and test with the regression function.
    wm_affine_updated = affine_assignment.affine_assignment(
        reference_img = cerebel_ref_path,
        source_img = t1_source_path,

        results_path = results_path, # deprecated in function - need to update there.

        # should I also update qform? If yes, should set qform (of source image) equal to sform (of reference image) - since our reference image is the cerebellum-only image (coregistered), so it doesn't have qform
        update_sform = True # just in case
    )
    

    print(f'{subj_id} {week} wm_native affine updated!')
    nib.save(wm_affine_updated, f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/c2{subj_id}_{week}_T1.nii')
"""

the issue there if with trying to get the path to the source_name, so we can either:
(a) figure this out (get name of nifti img) or
(b) just manually put in the name in the function

I think I will make the output just the image, and then save the image in the tutorial (in this loop)

This has been fixed, I think?